<a href="https://colab.research.google.com/github/leejunho12316/HonGongMachine/blob/main/%EC%95%99%EC%83%81%EB%B8%94_%EB%A6%AC%ED%8A%B8%EB%A6%AC%EB%B2%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip -q install langchain openai tiktoken langchain-community rank_bm25 sentence-transformers chromadb pypdf langchain-huggingface langchain_openai

In [4]:
import tiktoken
import openai
from typing import List, Dict, Optional, Any
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma, FAISS
from langchain_community.document_loaders import TextLoader, PyPDFLoader, WebBaseLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers  import BM25Retriever

In [5]:
model_huggingface = HuggingFaceEmbeddings(model_name='BAAI/bge-m3')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

#데이터 준비

In [6]:
## pdf 파일로드 하고 쪼개기
loader = PyPDFLoader('https://wdr.ubion.co.kr/wowpass/img/event/gsat_170823/gsat_170823.pdf')
pages = loader.load_and_split()

## chunk로 쪼개기
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0)
texts = text_splitter.split_documents(pages)

#Retriever 준비

In [7]:
bm25_retriever = BM25Retriever.from_documents(texts)

chroma_vector = Chroma.from_documents(texts, model_huggingface)
chroma_retriever = chroma_vector.as_retriever(search_kwargs={'k':2})

#앙상블 - 점수

각 Retriever의 문서별 점수 X 가중치

Ensemble Retriever가 이렇게 작동함

In [8]:
class EnsembleRetriever:
    """
    BM25 + Dense retriever 앙상블.
    Reciprocal Rank Fusion (RRF) 알고리즘으로 점수 결합.
    """
    def __init__(self, retrievers: List, weights: Optional[List[float]] = None, c: int = 60, id_key: Optional[str] = None):
        assert len(retrievers) == len(weights), "retrievers와 weights 길이가 다릅니다."
        self.retrievers = retrievers
        self.weights = weights
        self.c = c
        self.id_key = id_key

    def _get_doc_id(self, doc: Document) -> str:
        """문서 중복 판단용 ID 추출"""
        if self.id_key and self.id_key in doc.metadata:
            return str(doc.metadata[self.id_key])
        return doc.page_content[:150]

    def _fuse_results(self, all_results: List[List[Document]]) -> List[Document]:
        """Reciprocal Rank Fusion 계산"""
        scores: Dict[str, Dict[str, Any]] = {}
        for i, docs in enumerate(all_results):
            weight = self.weights[i] if self.weights else 1.0
            for rank, doc in enumerate(docs):
                doc_id = self._get_doc_id(doc)
                score = weight / (rank + 1 + self.c)
                if doc_id not in scores:
                    scores[doc_id] = {"doc": doc, "score": score}
                else:
                    scores[doc_id]["score"] += score

        sorted_docs = sorted(scores.values(), key=lambda x: x["score"], reverse=True)
        return [item["doc"] for item in sorted_docs]

    def invoke(self, query: str, k: int = 4) -> List[Document]:
        """모든 retriever에서 검색 후 점수 융합"""
        all_results = []
        for retriever in self.retrievers:
            if hasattr(retriever, "get_relevant_documents"):
                docs = retriever.get_relevant_documents(query)
            elif hasattr(retriever, "invoke"):
                docs = retriever.invoke(query)
            else:
                raise AttributeError(f"{retriever} does not support retrieval.")
            all_results.append(docs[:k])

        fused_docs = self._fuse_results(all_results)
        return fused_docs[:k]

In [10]:
ensemble_retriever = EnsembleRetriever(
    retrievers = [bm25_retriever, chroma_retriever],
    weights = [0.2, 0.8]
)

docs = ensemble_retriever.invoke('삼성전자의 코카콜라는?')

In [13]:
for doc in docs:
  print(doc)
  print('-' * 50)

page_content='2
01 삼성전자 기업분석
(Samsung Electronics Co., Ltd)
Ⅰ 기업 일반 
1  기업개요
1) 기업소개 
본사주소 경기도 수원시 영통구 삼성로 129(매탄동 416)
사업분야 삼성그룹의 대표 기업으로 휴대폰, 정보통신기기, 반도체, TV 등을 생산 판매하는 제조업체
홈페이지 www.samsung.com/sec 구분 전기전자 대기업  
설립일 1961년 07월 01일 대표이사 권오현 
총자산1) 244조 매출액2) 200조
임직원수 95,374명 
∙ 1975년 1월 주식시장 상장
∙ 1984년 2월 삼성전자공업주식회사->삼성전자주식회사로 사명 변경 
∙ CE(Consumer Electronics), IM(Information technology & Mobile communications), DS(Device Solutions) 
3개의 부문으로 나누어 독립 경영.
부문 제품
CE TV, 모니터, 냉장고, 세탁기, 에어컨, 프린터, 의료기기 등' metadata={'creator': 'nPDF (pdftk 1.41)', 'page_label': '1', 'creationdate': '2017-08-16T00:21:02-08:00', 'page': 0, 'total_pages': 27, 'source': 'https://wdr.ubion.co.kr/wowpass/img/event/gsat_170823/gsat_170823.pdf', 'producer': 'itext-paulo-155 (itextpdf.sf.net-lowagie.com)', 'moddate': '2017-08-16T00:21:02-08:00'}
--------------------------------------------------
page_content='11
Ⅱ 기업 상세 분석
1  사업분야(내용)
Q1 삼성전자의 대표적 사업분야에 대해 설명할 수 있습니까 ?
A
 삼성전자는 크게 CE(Consumer Electronics) 사업부문, IM(In